In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 작업 모니터링 또는 취소

[워크로드 페이지](https://quantum.cloud.ibm.com/workloads)에서 워크로드 목록을 확인할 수 있습니다.

## 작업 상태 보기
[워크로드 테이블](https://quantum.cloud.ibm.com/workloads)로 이동하여 상태 열에서 작업이 완료되었는지 또는 실패했는지 확인합니다.

## 남은 사용량 보기
[인스턴스 테이블](https://quantum.cloud.ibm.com/instances)로 이동하여 남은 사용량을 확인하려는 플랜과 연관된 탭을 선택합니다. 플랜에서 사용된 총 시간과 남은 총 시간이 표시됩니다.

## 제출된 작업 및 워크로드 수에 대한 지표 보기
[분석 페이지](https://quantum.cloud.ibm.com/analytics)로 이동하여 제출된 총 작업 수와 함께 배치 워크로드 및 세션 워크로드의 수를 확인합니다. 분석 페이지는 소유하거나 관리하는 계정에 대해서만 볼 수 있습니다.

## 작업 모니터링
작업 인스턴스를 사용하여 작업 상태를 확인하거나 적절한 명령을 호출하여 결과를 검색합니다.

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | 작업이 완료된 직후 작업 결과를 검토합니다. 작업 결과는 작업이 완료된 후에 사용할 수 있습니다. 따라서 job.result()는 작업이 완료될 때까지 블로킹 호출입니다.                               |
| job.job\_id()                 | 해당 작업을 고유하게 식별하는 ID를 반환합니다. 나중에 작업 결과를 검색하려면 작업 ID가 필요합니다. 따라서 나중에 검색할 작업의 ID를 저장해 두는 것이 좋습니다. |
| job.status()                  | 작업 상태를 확인합니다.                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | 이전에 제출한 작업을 검색합니다. 이 호출에는 작업 ID가 필요합니다.                                                                                                                                      |

<span id="retrieve-later"></span>
## 나중에 작업 결과 검색
`service.job(\<job\_id>)`를 호출하여 이전에 제출한 작업을 검색합니다. 작업 ID가 없거나 여러 작업을 한 번에 검색하려는 경우(퇴역한 QPU(양자 처리 장치)의 작업 포함), 선택적 필터를 사용하여 `service.jobs()`를 호출합니다. [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs)를 참조하세요.

> **Note:** `service.jobs()`는 더 이상 사용되지 않는 `qiskit-ibm-provider` 패키지에서 실행된 작업도 반환합니다. 이전(역시 더 이상 사용되지 않는) `qiskit-ibmq-provider` 패키지로 제출된 작업은 더 이상 사용할 수 없습니다.

### 예시
이 예시는 `my_backend`에서 실행된 가장 최근 런타임 작업 10개를 반환합니다.

In [ ]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Grover's algorithm](/docs/tutorials/grovers-algorithm) tutorial.
    - Learn more about [Sampler execution spans](/docs/guides/sampler-input-output#execution-spans)
</Admonition>